# الدرس 01 - مقدمة لوكلاء الذكاء الاصطناعي

مرحبًا بك في الدرس الأول من دورة **وكلاء الذكاء الاصطناعي للمبتدئين**!

الوكيل الذكي هو برنامج يستخدم نموذج لغة كبير (LLM) كمحرك تفكير له ويمكنه اتخاذ *إجراءات* في العالم الحقيقي — مثل استدعاء واجهات برمجة التطبيقات، أو الاستعلام من قواعد البيانات، أو تشغيل الكود — لتحقيق هدف نيابةً عن المستخدم.

في هذا الدفتر ستبني وكيلك الأول: **وكيل سفر** يوصي بوجهات العطلات. خلال الطريق ستتعلم كيف:

1. الاتصال بخدمة Microsoft Foundry Agent باستخدام **إطار عمل وكيل مايكروسوفت**.
2. إعطاء الوكيل **أداة** — دالة Python بسيطة يمكنه استدعاؤها.
3. تشغيل الوكيل وفحص استجابته.
4. بث استجابة الوكيل رمزًا برمز.


## الإعداد

قبل تشغيل هذا الدفتر، تأكد من أن لديك:

1. **مشروع Microsoft Foundry** مع نموذج دردشة مُنشر (على سبيل المثال `gpt-4.1-mini`).
2. **تسجيل الدخول باستخدام Azure CLI** — شغّل الأمر `az login` في الطرفية الخاصة بك.
3. **ضبط متغيرات البيئة المطلوبة:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطة النهاية لمشروع Microsoft Foundry الخاص بك.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج الذي تم نشره.

الخلية أدناه تقوم بتثبيت حزم بايثون التي تحتاجها.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## إنشاء وكيلك الأول

يحتاج الوكيل إلى شيئين:

- **تعليمات** تخبره *من هو* و*كيف يتصرف* (موجه النظام).
- **أدوات** — دوال بايثون مزينة بـ `@tool` يمكن للوكيل استدعاؤها لاسترجاع معلومات أو تنفيذ إجراءات.

أدناه نُعرّف أداة بسيطة تُرجع قائمة بوجهات السفر الشهيرة. سيستخدم الوكيل هذه الأداة عندما يطلب المستخدم توصيات سفر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ردود متدفقة

لتجربة أكثر تفاعلية يمكنك **تدفق** رد الوكيل. بدلاً من الانتظار للحصول على الرد الكامل، يُخرج الوكيل أجزاءً من النص بينما يتم توليدها. هذا مفيد بشكل خاص في واجهات الدردشة حيث ترغب في عرض المخرجات في الوقت الحقيقي.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيف:

- **إنشاء مزود** يتصل بخدمة وكيل Microsoft Foundry عبر `FoundryChatClient`.
- **تعريف أداة** باستخدام مزخرف `@tool` حتى يتمكن الوكيل من استدعاء دوال بايثون الخاصة بك.
- **تشغيل الوكيل** مع رسالة مستخدم وطباعة استجابته.
- **بث الاستجابات** للإخراج في الوقت الحقيقي.

في الدرس التالي سنستكشف أطر العمل الوكيلة بمزيد من العمق ونتعلم كيفية تزويد الوكلاء بأدوات أكثر قوة وقدرات استدلال متعددة الخطوات.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
